In [0]:
%pip install azure-data-tables

from azure.data.tables      import TableServiceClient
from azure.core.credentials import AzureNamedKeyCredential

# ── Config ────────────────────────────────────────────────────────────────────
STORAGE_ACCOUNT = "storageproject12026"
STORAGE_KEY     = spark.conf.get("spark.hadoop.fs.azure.account.key.storageproject12026.dfs.core.windows.net")
BRONZE_ROOT     = "abfss://bronze@storageproject12026.dfs.core.windows.net/SalesLT"

# Tables to copy: { Azure Table Storage name : Bronze folder name }
TABLES = {
    "MarketingCampaigns":    "MarketingCampaigns",
    "MarketingLeads":        "MarketingLeads",
    "MarketingInteractions": "MarketingInteractions",
}

# Connect to Azure Table Storage
credential = AzureNamedKeyCredential(STORAGE_ACCOUNT, STORAGE_KEY)
service    = TableServiceClient(
    endpoint   = f"https://{STORAGE_ACCOUNT}.table.core.windows.net",
    credential = credential
)

print("✓ Config loaded")
print(f"  Storage account : {STORAGE_ACCOUNT}")
print(f"  Bronze root     : {BRONZE_ROOT}")
print(f"  Tables to copy  : {list(TABLES.keys())}")

# Test connection immediately
tables = list(service.list_tables())
print(f"\n✓ Connection successful — {len(tables)} tables found:")
for t in tables:
    print(f"  → {t.name}")

In [0]:
 #─────────────────────────────────────────────────────────────────────────────
# CELL 1 — HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────
 
def read_table_storage(table_name: str) -> list:
    """Read all entities from an Azure Table Storage table."""
    table_client = service.get_table_client(table_name)
    entities     = list(table_client.list_entities())
    print(f"  ✓ Read {len(entities):,} rows from {table_name}")
    return entities
 
 
def clean_entity(entity: dict) -> dict:
    """
    Clean a Table Storage entity for Spark:
    - Remove Azure internal keys (PartitionKey, RowKey, Timestamp, etag)
    - Parse nested JSON strings back to strings (keep as JSON for Bronze)
    - Convert booleans and numbers
    """
    skip_keys = {"PartitionKey", "RowKey", "Timestamp", "etag"}
    cleaned   = {}
    for k, v in entity.items():
        if k in skip_keys:
            continue
        # Keep nested JSON as string in Bronze (raw layer)
        if isinstance(v, (dict, list)):
            cleaned[k] = json.dumps(v, default=str)
        elif hasattr(v, 'isoformat'):
            # Convert datetime objects to string
            cleaned[k] = v.isoformat()
        else:
            cleaned[k] = v
    return cleaned
 
 
def entities_to_spark_df(entities: list):
    """Convert list of Table Storage entities to a Spark DataFrame."""
    if not entities:
        return spark.createDataFrame([], StructType([]))
 
    # Clean all entities
    cleaned = [clean_entity(e) for e in entities]
 
    # Convert to pandas first (handles mixed types gracefully)
    pdf = pd.DataFrame(cleaned)
 
    # Convert all columns to string for Bronze (raw layer — no type enforcement)
    for col in pdf.columns:
        pdf[col] = pdf[col].astype(str).replace("None", "").replace("nan", "")
 
    return spark.createDataFrame(pdf)
 
 
def write_to_bronze(df, folder_name: str) -> str:
    """Write Spark DataFrame to Bronze as Parquet."""
    path = f"{BRONZE_ROOT}/{folder_name}/"
    df.write \
      .format("parquet") \
      .mode("overwrite") \
      .save(path)
    return path
 
 
print("✓ CELL 1 — Helper functions ready")
 

In [0]:
import pandas as pd  

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — COPY ALL TABLES TO BRONZE
# ─────────────────────────────────────────────────────────────────────────────
 
results = {}
 
for table_name, folder_name in TABLES.items():
    print(f"\n── Processing {table_name} ──────────────────────────")
 
    try:
        # Step 1: Read from Azure Table Storage
        entities = read_table_storage(table_name)
 
        if not entities:
            print(f"  ⚠  {table_name} is empty — skipping")
            continue
 
        # Step 2: Convert to Spark DataFrame
        df = entities_to_spark_df(entities)
        print(f"  ✓ Converted to DataFrame — {df.count():,} rows | {len(df.columns)} cols")
 
        # Step 3: Write to Bronze ADLS as Parquet
        path = write_to_bronze(df, folder_name)
        print(f"  ✓ Written to Bronze → {path}")
 
        results[table_name] = {
            "rows":    df.count(),
            "columns": len(df.columns),
            "path":    path,
            "status":  "✓ Success"
        }
 
    except Exception as e:
        print(f"  ✗ Error: {e}")
        results[table_name] = {
            "rows":   0,
            "status": f"✗ Failed: {e}"
        }
 
print("\n✓ CELL 2 — All tables processed")

In [0]:
print("── Verifying Bronze files ───────────────────────────────")
for table_name, folder_name in TABLES.items():
    path = f"{BRONZE_ROOT}/{folder_name}/"
    try:
        df = spark.read.format("parquet").load(path)
        print(f"\n  [{folder_name}]  {df.count():,} rows | {len(df.columns)} cols")
        df.show(3, truncate=50)
    except Exception as e:
        print(f"  ✗ [{folder_name}] could not read: {e}")
 
print("\n✓ CELL 3 — Verification complete")